# 본 분석 전 기술통계 및 전처리 필요 항목 점검

본 분석에선 실제 전처리를 수행하지 않고,\
**분석에 들어가기 앞서 어떤 전처리가 필요한지 확인하는 진단용 기술통계**입니다.

## 1. 중점 확인 항목
| 기준 | 확인 내용 |
|---|---|
| 결측 | 컬럼별 결측치 수와 비율 |
| 중복 | 완전 중복 행, `appid` 중복 |
| 공백 | 문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백 |
| 타입 | 컬럼별 dtype, 고유값 수, 구조형 문자열 여부 |
| 이상치 | 수치형 컬럼의 IQR 기준 이상치 후보 |
| 조인 가능성 | `appid` 기준 테이블 간 매칭률 |


## 점검 범위
- `steam_indie_9692_202604281029.csv`


# 2. 라이브러리 호출

In [15]:
from pathlib import Path
import ast
import json
import re
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 보기 옵션
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# 3. 파일 경로 설정 및 데이터 읽기

In [16]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정
# db에서 바로 불러오는법을 모름
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 원천/소스 파일이 들어있는 폴더
DATA_DIR = ROOT / "data" / "processed"

# 파일 경로

MAIN_PATH = DATA_DIR / "steam_indie_9692_202604281029.csv"


# 경로 확인
# 파일이 없으면 이후 read_csv 단계에서 에러가 나므로, 먼저 exists() 결과를 확인한다.
print("ROOT             =", ROOT)
print("DATA_DIR         =", DATA_DIR)
print("MAIN_PATH  =", MAIN_PATH)


print()
print("main_df exists  :", MAIN_PATH.exists())


# 데이터 읽기
# 여기서는 원본을 바로 전처리하지 않고, 파일을 읽어온 뒤 별도 확인용 복사본을 만든다.
main_df = pd.read_csv(MAIN_PATH)


# 크기 확인
print()
print("main_df shape  :", main_df.shape)

ROOT             = C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR         = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
MAIN_PATH  = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_202604281029.csv

main_df exists  : True

main_df shape  : (9692, 14)


## 4. 실제 사용하는 기술통계 함수

In [17]:
def check_basic_info(df, df_name, exclude_cols=None):
    """행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [18]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """appid처럼 기준 키로 쓸 컬럼의 중복 여부를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    duplicate_count = df_copied[col_name].duplicated().sum()

    print('전체 행 수:', len(df_copied))
    print(f'{col_name} 고유 개수:', df_copied[col_name].nunique(dropna=True))
    print(f'중복 {col_name} 개수:', duplicate_count)

    if duplicate_count > 0:
        print()
        print('[중복 상위 값]')
        dup_summary = df_copied[col_name].value_counts(dropna=False).reset_index()
        dup_summary.columns = [col_name, '등장 횟수']
        display(dup_summary[dup_summary['등장 횟수'] > 1].head(top_n))
    else:
        print('중복 값이 없습니다.')

In [19]:
def check_string_space_summary(df, df_name, top_n=30):
    """문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 문자열 공백/빈값 확인")
    print(f"{'='*80}")

    object_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

    if len(object_cols) == 0:
        print("문자열 컬럼이 없습니다.")
        return

    rows = []

    for col in object_cols:
        temp = df[col]
        temp_str = temp.dropna().astype(str)

        empty_count = temp_str.str.strip().eq("").sum()
        leading_trailing_count = temp_str.ne(temp_str.str.strip()).sum()
        multi_space_count = temp_str.str.contains(r"\s{2,}", regex=True).sum()

        rows.append({
            "컬럼명": col,
            "문자열 행 수": len(temp_str),
            "빈 문자열/공백값 개수": empty_count,
            "앞뒤 공백 개수": leading_trailing_count,
            "연속 공백 포함 개수": multi_space_count
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(
        by=["빈 문자열/공백값 개수", "앞뒤 공백 개수", "연속 공백 포함 개수"],
        ascending=False
    )

    display(summary_df.head(top_n))

In [20]:
def check_numeric_summary(df, df_name, cols=None):
    """수치형 컬럼의 기본 기술통계, 왜도, 첨도를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 수치형 기술통계")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    summary_df = df[numeric_cols].describe().T
    summary_df["결측치 개수"] = df[numeric_cols].isnull().sum()
    summary_df["왜도"] = df[numeric_cols].skew(numeric_only=True)
    summary_df["첨도"] = df[numeric_cols].kurt(numeric_only=True)

    display(summary_df)

In [21]:
def check_iqr_outlier_summary(df, df_name, cols=None, top_n=30):
    """IQR 기준으로 이상치 후보가 많은 수치형 컬럼을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 IQR 기준 이상치 후보 확인")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    rows = []

    for col in numeric_cols:
        temp = df[col].dropna()

        if len(temp) == 0:
            continue

        q1 = temp.quantile(0.25)
        q3 = temp.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_count = ((temp < lower) | (temp > upper)).sum()

        rows.append({
            "컬럼명": col,
            "확인 행 수": len(temp),
            "결측치 개수": df[col].isnull().sum(),
            "최솟값": temp.min(),
            "Q1": q1,
            "중앙값": temp.median(),
            "Q3": q3,
            "최댓값": temp.max(),
            "IQR": iqr,
            "하한 기준": lower,
            "상한 기준": upper,
            "이상치 후보 개수": outlier_count,
            "이상치 후보 비율(%)": round(outlier_count / len(temp) * 100, 2)
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values("이상치 후보 개수", ascending=False)

    display(summary_df.head(top_n))

In [22]:
def check_category_summary(df, df_name, col_name, top_n=10):
    """범주형 컬럼의 값 분포를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    print("전체 행 수:", len(df))
    print(f"{col_name} 고유값 개수(결측 포함):", df[col_name].nunique(dropna=False))
    print()

    display(summary_df.head(top_n))

In [23]:
def check_datetime_parse_summary(df, df_name, col_name):
    """실제 전처리 컬럼을 생성하지 않고, 변환 성공/실패 여부와 날짜 범위만 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 날짜 변환 가능성 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    parsed = pd.to_datetime(df[col_name], errors="coerce")

    summary_df = pd.DataFrame({
        "항목": [
            "전체 행 수", 
            "원본 결측치 수", 
            "날짜 변환 실패 수", 
            "날짜 변환 성공 수", 
            "최소 날짜", 
            "최대 날짜"]
        ,
        "값": [
            len(df),
            df[col_name].isnull().sum(),
            parsed.isnull().sum(),
            parsed.notnull().sum(),
            parsed.min(),
            parsed.max()
        ]
    })

    display(summary_df)

    print("\n연도별 분포")
    display(parsed.dt.year.value_counts(dropna=False).sort_index().reset_index().rename(
        columns={"release_date": "연도", "count": "개수"}
    ))

In [24]:

def check_list_string_summary(df, df_name, col_name, top_n=20):
    """문자열로 저장된 리스트형 컬럼을 파싱해 항목 분포를 확인하는 함수"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 리스트형 문자열 파싱 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    def safe_parse_list(x):
        if pd.isna(x):
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []

    parsed_series = df[col_name].apply(safe_parse_list)

    parse_fail_count = ((df[col_name].notna()) & (parsed_series.apply(len) == 0)).sum()
    list_len = parsed_series.apply(len)

    print("전체 행 수:", len(df))
    print("파싱 실패 또는 빈 리스트 수:", parse_fail_count)
    print("평균 항목 수:", round(list_len.mean(), 2))
    print("최소 항목 수:", list_len.min())
    print("최대 항목 수:", list_len.max())

    exploded = parsed_series.explode()
    summary_df = exploded.value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    display(summary_df.head(top_n))

## 5. `steam_indie_9692` 데이터 점검 실행

In [25]:
# 1) 기본 구조, 타입, 결측치 확인
check_basic_info(main_df, "steam_indie_9692")

# 2) 게임 단위 식별자인 appid 중복 확인
check_id_duplicates(main_df, "appid", "steam_indie_9692")

# 3) 문자열 컬럼의 공백/빈값 확인
check_string_space_summary(main_df, "steam_indie_9692")


steam_indie_9692의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9692
1,열 개수,14
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,str,9680,99.88,12,0.12,8139
appid,int64,9692,100.00,0,0.00,9692
name,str,9692,100.00,0,0.00,9688
total_reviews,int64,9692,100.00,0,0.00,1427
positive,int64,9692,100.00,0,0.00,1346
release_date,str,9692,100.00,0,0.00,1008
negative,int64,9692,100.00,0,0.00,601
ccu,int64,9692,100.00,0,0.00,294
price,int64,9692,100.00,0,0.00,282
genres,str,9692,100.00,0,0.00,250


[상위 5행]


,appid,name,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access
0,899770,Last Epoch,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False
1,251570,7 Days to Die,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False
2,1116170,CyberCorp,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False
3,1326470,Sons Of The Forest,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False
4,2186680,"Warhammer 40,000: Rogue Trader","10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False



steam_indie_9692의 appid 값 중복 확인
전체 행 수: 9692
appid 고유 개수: 9692
중복 appid 개수: 0
중복 값이 없습니다.

steam_indie_9692의 문자열 공백/빈값 확인


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
4,developers,9680,0,11,5
0,name,9692,0,7,17
1,owners,9692,0,0,0
2,genres,9692,0,0,0
3,release_date,9692,0,0,0


In [26]:
# 4) 수치형 컬럼 기술통계
# appid는 식별자이므로 수치형 기술통계 해석 대상에서는 제외합니다.
numeric_analysis_cols = [
    "positive",
    "negative",
    "price",
    "ccu",
    "total_reviews",
    "owners_lower"
]

check_numeric_summary(
    main_df,
    "steam_indie_9692",
    cols=numeric_analysis_cols
)

# 5) IQR 기준 이상치 후보 확인
# 이상치는 삭제 목적이 아니라, 분포 특성과 극단값 존재 여부를 확인하기 위한 것입니다.
check_iqr_outlier_summary(
    main_df,
    "steam_indie_9692",
    cols=numeric_analysis_cols
)


steam_indie_9692의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,9692.0,696.681903,6750.596063,0.0,15.0,34.0,127.0,327889.0,0,27.696358,996.277080
negative,9692.0,97.027652,1343.304549,0.0,2.0,6.0,22.0,106084.0,0,57.683397,4171.168253
price,9692.0,883.707078,1058.550707,0.0,299.0,599.0,1199.0,19999.0,0,9.899780,165.728969
ccu,9692.0,35.946038,928.321453,0.0,0.0,0.0,1.0,83936.0,0,78.121949,6915.752078
total_reviews,9692.0,793.709554,7662.793747,10.0,18.0,41.0,152.0,370046.0,0,27.808549,990.715919
owners_lower,9692.0,32220.387949,333407.472154,0.0,0.0,0.0,20000.0,20000000.0,0,37.329546,1754.736105



steam_indie_9692의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
3,ccu,9692,0,0,0.0,0.0,1.0,83936,1.0,-1.5,2.5,1864,19.23
4,total_reviews,9692,0,10,18.0,41.0,152.0,370046,134.0,-183.0,353.0,1501,15.49
0,positive,9692,0,0,15.0,34.0,127.0,327889,112.0,-153.0,295.0,1500,15.48
1,negative,9692,0,0,2.0,6.0,22.0,106084,20.0,-28.0,52.0,1448,14.94
5,owners_lower,9692,0,0,0.0,0.0,20000.0,20000000,20000.0,-30000.0,50000.0,689,7.11
2,price,9692,0,0,299.0,599.0,1199.0,19999,900.0,-1051.0,2549.0,229,2.36


In [27]:
# 6) 주요 범주형 컬럼 확인
check_category_summary(main_df, "steam_indie_9692", "owners", top_n=20)
check_category_summary(main_df, "steam_indie_9692", "is_f2p")
check_category_summary(main_df, "steam_indie_9692", "is_early_access")


steam_indie_9692의 owners 범주 확인
전체 행 수: 9692
owners 고유값 개수(결측 포함): 11



,owners,개수,비율(%)
0,"0 .. 20,000",7228,74.58
1,"20,000 .. 50,000",1209,12.47
2,"50,000 .. 100,000",566,5.84
3,"100,000 .. 200,000",319,3.29
4,"200,000 .. 500,000",237,2.45
5,"500,000 .. 1,000,000",79,0.82
6,"1,000,000 .. 2,000,000",34,0.35
7,"2,000,000 .. 5,000,000",11,0.11
8,"10,000,000 .. 20,000,000",5,0.05
9,"5,000,000 .. 10,000,000",3,0.03



steam_indie_9692의 is_f2p 범주 확인
전체 행 수: 9692
is_f2p 고유값 개수(결측 포함): 1



,is_f2p,개수,비율(%)
0,False,9692,100.0



steam_indie_9692의 is_early_access 범주 확인
전체 행 수: 9692
is_early_access 고유값 개수(결측 포함): 1



,is_early_access,개수,비율(%)
0,False,9692,100.0


In [28]:
# 7) 출시일 변환 가능성 확인
check_datetime_parse_summary(main_df, "steam_indie_9692", "release_date")

# 8) 장르 리스트형 문자열 파싱 가능성 확인
check_list_string_summary(main_df, "steam_indie_9692", "genres", top_n=20)


steam_indie_9692의 release_date 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,9692
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,9692
4,최소 날짜,2023-01-01 00:00:00
5,최대 날짜,2025-12-27 00:00:00



연도별 분포


,연도,개수
0,2023,3498
1,2024,4180
2,2025,2014



steam_indie_9692의 genres 리스트형 문자열 파싱 확인
전체 행 수: 9692
파싱 실패 또는 빈 리스트 수: 0
평균 항목 수: 3.12
최소 항목 수: 1
최대 항목 수: 10


,genres,개수,비율(%)
0,Indie,9687,99.95
1,Adventure,4807,49.60
2,Casual,4111,42.42
3,Action,4093,42.23
4,Simulation,2503,25.83
5,RPG,2194,22.64
6,Strategy,2005,20.69
7,Sports,341,3.52
8,Racing,301,3.11
9,Massively Multiplayer,124,1.28


# 6.기술통계 보고서

## 1. 데이터 구조
| 항목 | 값 |
|---|---:|
| 행 수 | 9,692 |
| 열 수 | 14 |
| 완전 중복 행 수 | 0 |
| `appid` 고유값 수 | 9,692 |
| `appid` 중복 수 | 0 |

현재 데이터는 게임 1개당 1행 구조

## 2. 타입

| 컬럼 유형 | 컬럼 |
|---|---|
| 식별자 | `appid` |
| 게임 기본 정보 | `name`, `developers`, `release_date` |
| 시장/성과 지표 | `owners`, `owners_lower`, `positive`, `negative`, `total_reviews`, `ccu` |
| 가격 정보 | `price` |
| 장르 정보 | `genres` |
| 제외 조건 확인용 | `is_f2p`, `is_early_access` |

`release_date`는 현재 문자열이므로 출시일 기준 분석을 위해 날짜형 변환이 필요하다.\
`genres`는 문자열 형태의 리스트이므로 장르별 분석을 위해 리스트 파싱이 필요하다.


## 3. 결측치
| 컬럼 | 결측치 수 | 결측치 비율 |
|---|---:|---:|
| `developers` | 12 | 0.12% |

`developers`에서만 소량 확인 전체 분석에는 큰 영향을 주지 않을 가능성이 높음\
다만 개발사 기준 집계, 인디 여부를 개발사명으로 추가 검토할 경우에는 해당 결측값을 확인 필요

## 4. 중복
완전 중복 행과 `appid` 중복은 모두 0개로 확인\
따라서 중복 제거는 현재 필수 전처리로 보이지 않는다.

## 5. 문자열 공백/빈값
| 컬럼 | 빈 문자열 수 | 앞뒤 공백 수 | 연속 공백 수 |
|---|---:|---:|---:|
| `name` | 0 | 7 | 17 |
| `owners` | 0 | 0 | 0 |
| `genres` | 0 | 0 | 0 |
| `release_date` | 0 | 0 | 0 |
| `developers` | 0 | 11 | 5 |

문자열 컬럼에서 빈 문자열은 확인되지 않았으나\
`name`, `developers`에서 일부 앞뒤 공백과 연속 공백이 확인\
게임명이나 개발사명 기준으로 그룹화할 경우 `str.strip()`과 연속 공백 정리를 고려할 필요가 있다.


## 6. 이상치 후보
| 컬럼 | 중앙값 | 최댓값 | IQR 기준 이상치 후보 수 | 이상치 후보 비율 |
|---|---:|---:|---:|---:|
| `positive` | 34 | 327,889 | 1,500 | 15.48% |
| `negative` | 6 | 106,084 | 1,448 | 14.94% |
| `price` | 599 | 19,999 | 229 | 2.36% |
| `ccu` | 0 | 83,936 | 1,864 | 19.23% |
| `total_reviews` | 41 | 370,046 | 1,501 | 15.49% |
| `owners_lower` | 0 | 20,000,000 | 689 | 7.11% |

리뷰 수, 동시접속자 수, 소유자 수는 강한 우측 치우침을 보인다.\
다만, 이는 데이터 오류로 보기보단, Steam 인디게임 시장에서 소수의 대형 흥행작과 다수의 소규모 게임이 함께 존재하기 때문에 나타나는 자연스러운 구조로 해석하는 것이 타당하다.\
따라서 이상치 후보를 단순 삭제하기보다는, 로그 변환 또는 규모 구간화 같은 방식으로 다루는 것이 좋다.


## 7. 주요 범주형 변수

### 7-1. `owners` 분포
| owners 구간 | 개수 | 비율 |
|---|---:|---:|
| 0 .. 20,000 | 7,228 | 74.58% |
| 20,000 .. 50,000 | 1,209 | 12.47% |
| 50,000 .. 100,000 | 566 | 5.84% |
| 100,000 .. 200,000 | 319 | 3.29% |
| 200,000 .. 500,000 | 237 | 2.45% |
| 500,000 .. 1,000,000 | 79 | 0.82% |
| 1,000,000 .. 2,000,000 | 34 | 0.35% |
| 2,000,000 .. 5,000,000 | 11 | 0.11% |
| 10,000,000 .. 20,000,000 | 5 | 0.05% |
| 5,000,000 .. 10,000,000 | 3 | 0.03% |
| 20,000,000 .. 50,000,000 | 1 | 0.01% |

### 7-2. `is_f2p`, `is_early_access`
| 컬럼 | 값 |
|---|---|
| `is_f2p` | 전체 `False` |
| `is_early_access` | 전체 `False` |

## 8. 날짜와 장르 구조

### 8-1. `release_date`
| 항목 | 값 |
|---|---|
| 최소 출시일 | 2023-01-01 |
| 최대 출시일 | 2025-12-27 |
| 날짜 변환 실패 수 | 0 |

날짜 변환 실패 없이 변환 가능

### 8-2. `genres`
| 장르 | 개수 | 비율 |
|---|---:|---:|
| Indie | 9,687 | 99.95% |
| Adventure | 4,807 | 49.60% |
| Casual | 4,111 | 42.42% |
| Action | 4,093 | 42.23% |
| Simulation | 2,503 | 25.83% |
| RPG | 2,194 | 22.64% |
| Strategy | 2,005 | 20.69% |
| Sports | 341 | 3.52% |
| Racing | 301 | 3.11% |
| Massively Multiplayer | 124 | 1.28% |

문자열 형태의 리스트로 저장되어 있으며, 파싱 실패 없이 리스트로 변환 가능

## 본 분석 전 필요한 전처리 후보

| 전처리 항목 | 필요 여부 | 이유 |
|---|---|---|
| `appid` 중복 제거 | 굳이? | 현재 중복 없음 |
| 결측치 제거/대체 | 굳이? | `developers` 결측만 소량 존재 |
| 문자열 공백 제거 | 필수 | `name`, `developers` 집계 안정성 확보 |
| `release_date` 날짜형 변환 | 필수 | 출시 연도/월/기간 분석에 필요 |
| `genres` 리스트 파싱 | 필수 | 장르별 분석에 필요 |
| 리뷰 수/소유자 수 로그 변환 | 선택 | 분포 치우침이 강하므로 통계/모델링 시 고려 |
| 가격 구간화 | 선택 | 가격대별 비교 분석 시 필요 |
| owners 구간 정리 | 선택 | 흥행 규모 구간 변수로 활용 가능 |

## 본 분석 확장 시 추가로 필요한 데이터

| 분석하고 싶은 내용 | 추가로 필요한 데이터 | 예상 조인 키 | 이유 |
|---|---|---|---|
| 리뷰 감성 분석 | 리뷰 본문, 추천/비추천, 작성 시점, 플레이타임? | `appid`, 필요 시 `recommendationid` | `steam_indie_9692`에는 리뷰 본문이 없으므로 감성/불만 원인 분석 불가 |
| 장르/태그 기반 세부 포지셔닝 | Steam 태그 데이터 | `appid` | 공식 장르보다 세부적인 유저 인식 태그 분석 가능 |
| 플레이 방식 분석 | `categories` 또는 별도 메타데이터 | `appid` | 싱글/멀티/협동/컨트롤러 지원 여부 등 확인 필요 |
| 초기 반응 추세 분석 | 리뷰 히스토리/일자별 리뷰 집계 | `appid`, 날짜 | D7, D30 과 같은 출시 직후 반응을 보려면 시간축 데이터 필요 |